# 44. 채도가 원판 테두리 검출에 영향을 줄까?

무릎 통과부터 바와 몸이 접촉할 때까지, 검증 선수 6명의 같은 56장을 비교합니다. 관심 지점은 수동 원판 박스의 **아랫변**과 직접 표시한 **허브 높이**입니다. 원본과 화면 전체의 채도를 줄인 장면·높인 장면을 나란히 봅니다. 원판과 배경의 색이 함께 바뀌므로 어느 영역이 원인인지는 이 실험만으로 구분하기 어렵습니다.

VS Code에서 개인 프로젝트의 `.venv` 커널을 선택한 뒤 위에서 아래로 실행하세요. **2단계에서 입력 이미지를 눈으로 확인한 뒤 3단계에서 추론**합니다. 추론은 기존 가중치로 Mac CPU에서 실행합니다.

## 1. 가중치와 프레임 확인

원판 베이스라인 `best.pt`, 입력 640, 후보 하한 0.001, 직접 검출 기준 0.10을 사용합니다. 각 영상에서 0번부터 현재 프레임 직전까지의 원본 후보를 세 조건이 공유합니다. 시간 연결의 허용 반경은 직전 박스 너비의 0.5배이며, 마지막 직접 검출 후 최대 두 프레임까지 연결합니다.

In [ ]:
from datetime import datetime
from hashlib import sha256
from pathlib import Path
import json
import sys

# VS Code나 Jupyter에서 열어도 현재 실험 브랜치의 모듈을 사용합니다.
start = Path(globals().get("__vsc_ipynb_file__", Path.cwd())).resolve()
root = next(p for p in (start, *start.parents) if (p / "pyproject.toml").is_file())
sys.path.insert(0, str(root))

import matplotlib.pyplot as plt
import pandas as pd
import ultralytics
from IPython.display import display
from wlcv.evaluation.plate_resolution import WEIGHTS, review_targets
from wlcv.evaluation.plate_saturation import FACTORS, check_original, paired_summary, run_review, show_frame

targets = review_targets()
weights_hash = sha256(WEIGHTS.read_bytes()).hexdigest()
reference = root / "outputs/plate_resolution/2026-11-24"
if (reference / "settings.json").exists():
    saved_hash = json.loads((reference / "settings.json").read_text())["weights_sha256"]
    if saved_hash != weights_hash:
        raise ValueError("33번 실험과 가중치가 다릅니다. 원판 베이스라인 best.pt를 확인하세요.")
print(f"선수 {targets.person_id.nunique()}명 · {len(targets)}장 · {WEIGHTS.name} · CPU")
display(targets.groupby(["person_id", "video_id"]).frame_index.agg(["min", "max", "count"])
        .rename(columns={"min": "무릎 통과", "max": "접촉", "count": "프레임 수"}))

## 2. 추론 전에 세 입력을 눈으로 비교

왼쪽은 디코딩한 원본, 가운데는 채도 0.5배, 오른쪽은 1.5배입니다. HSV에서 색상과 밝기를 고정하고 채도만 바꿉니다. 노란 점선은 수동 박스, 흰 십자는 직접 찍은 허브입니다. 두 사례의 색이 있는 부분과 어두운 테두리를 살펴보세요. 아래 목록을 1단계 표 안의 다른 영상·프레임으로 바꿀 수 있습니다.

In [ ]:
examples = [("a03_sn_f_01", 19), ("b02_sn_h_02", 35)]
for video, frame in examples:
    show_frame(targets, video, frame)
    plt.show()

## 3. 같은 직전 궤적에서 추론

진행 막대에는 원본 이력을 만드는 프레임도 포함됩니다. 각 비교에서 현재 장만 바꾸므로 변형한 장의 예측은 다음 장으로 이어지지 않습니다. 33번 실험의 로컬 비교표가 있으면 원본 56장의 선택 출처·매칭·박스 좌표·신뢰도·IoU를 프레임별로 다시 확인합니다.

In [ ]:
detail = run_review(targets)
reference_csv = reference / "paired_frames.csv"
if reference_csv.exists():
    check_original(detail, reference_csv)
    print("33번 입력 640과 원본 56장의 선택·좌표·점수를 대조했습니다.")
else:
    print("33번 paired_frames.csv가 없어 이전 실행과의 대조는 생략했습니다.")
print(f"평가 결과 {len(detail)}행 = {len(targets)}프레임 × {len(FACTORS)}조건")

## 4. 몇 장을 찾았고 테두리와 허브가 얼마나 움직였나?

IoU 0.50 이상을 정탐으로 셉니다. 첫 표는 조건별 전체 56장이 분모입니다. ‘박스 불일치’는 선택된 박스가 있지만 IoU가 낮은 경우, ‘출력 없음’은 선택할 박스가 없는 경우입니다. 둘 다 미탐에 포함됩니다.

두 번째 표는 선수별로 변형과 원본을 짝지어 비교합니다. 좌표 변화는 **공통 정탐**, 허브는 그중 **직접 판독한 공통 프레임**만 사용합니다. 아랫변·윗변·중심의 부호 있는 변화가 양수이면 변형에서 더 아래로 갔다는 뜻입니다. 절대오차 변화가 음수이면 수동값에 더 가까워졌습니다.

In [ ]:
counts = detail.assign(
    direct_hit=detail.match & detail.source.eq("high"),
    linked_hit=detail.match & detail.source.eq("linked"),
    wrong_box=~detail.match & detail.source.ne("missing"),
    no_box=detail.source.eq("missing"),
).groupby("factor")[["direct_hit", "linked_hit", "wrong_box", "no_box"]].sum().reindex(FACTORS)
counts["misses"] = counts.wrong_box + counts.no_box
display(counts.rename(columns={"direct_hit": "직접 정탐", "linked_hit": "연결 정탐",
                               "wrong_box": "박스 불일치", "no_box": "출력 없음", "misses": "미탐"}))

pairs = paired_summary(detail)
display(pairs.rename(columns={
    "person_id": "선수", "factor": "채도 배수", "frames": "전체", "common": "공통 정탐",
    "recovered": "복구", "lost": "손실", "direct_hits": "직접 정탐", "linked_hits": "연결 정탐",
    "hub_pairs": "허브 비교 장수", "delta_bottom_px": "아랫변 Δpx", "delta_top_px": "윗변 Δpx",
    "delta_center_px": "중심 Δpx", "delta_iou": "IoU 변화", "delta_abs_bottom_px": "아랫변 절대오차 Δpx",
    "delta_abs_hub_y_error_px": "허브 절대오차 Δpx",
}).round(2))

## 5. 선수별 아랫변 오차와 검출 이미지 확인

그래프의 0은 예측 아랫변과 수동 아랫변이 같은 위치라는 뜻입니다. 양수는 원판 아래까지 박스가 내려갔음을 뜻합니다. 접촉 프레임을 가로축 0에 맞췄습니다. 정탐이 사라진 지점에서 선이 끊깁니다. 아래의 세 장 비교에서 파란 실선은 각 이미지에서 선택한 예측 박스입니다.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(11, 6), layout="constrained", sharey=True)
for axis, (person, group) in zip(axes.flat, detail.groupby("person_id"), strict=True):
    contact = group.frame_index.max()
    for factor, marker in zip(FACTORS, ["o", "^", "x"], strict=True):
        rows = group[group.factor.eq(factor)].sort_values("frame_index")
        axis.plot(rows.frame_index - contact, rows.bottom_px.where(rows.match),
                  marker=marker, ms=4, label=f"S × {factor:g}")
    axis.axhline(0, color="grey", lw=0.8)
    axis.set_title(person)
    axis.grid(alpha=0.2)
axes[0, 0].legend(fontsize=8)
fig.supxlabel("Frames relative to contact (0 = contact)")
fig.supylabel("Predicted minus manual bottom edge (px, down +)")
plt.show()

for video, frame in examples:
    show_frame(targets, video, frame, detail)
    plt.show()

## 6. 결과를 로컬에 저장하고 다음 실험 결정

실행별 폴더에 프레임 표·선수별 짝비교·그래프·대표 오버레이와 조건을 저장합니다. 선수 여러 명에서 아랫변의 절대오차가 같은 방향으로 바뀌었는지, 허브 오차와 미탐도 함께 줄었는지 보세요. 한 선수에 집중되거나 차이가 재판독 변동 수준이면 현재 모델로 ONNX 비교를 진행합니다. 채도 변화에 따른 차이가 여러 선수에서 반복되면 학습의 `hsv_s`만 바꾸는 실험을 검토합니다.

In [ ]:
output = root / "outputs/plate_saturation_review" / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
output.mkdir(parents=True, exist_ok=False)
detail.to_csv(output / "frames.csv", index=False)
pairs.to_csv(output / "paired_by_person.csv", index=False)
counts.to_csv(output / "detections.csv")
fig.savefig(output / "bottom_edge.png", dpi=150)
for video, frame in examples:
    overlay = show_frame(targets, video, frame, detail)
    overlay.savefig(output / f"{video}_f{frame}.png", dpi=150)
    plt.close(overlay)
settings = dict(weights_sha256=weights_hash, ultralytics=ultralytics.__version__, factors=FACTORS,
                imgsz=640, device="cpu", candidate_conf=0.001, direct_conf=0.10,
                link_radius_ratio=0.5, max_frames_since_high=2, match_iou=0.50,
                history="original video before current frame", checked_reference=reference_csv.exists())
(output / "settings.json").write_text(json.dumps(settings, indent=2))
print(output.resolve())